Now, we can look into the synthesis of a conditional order, a.k.a. a pruning rule that is dependent to the takeoff times $t_i$ and $t_j$ directly.

In [1]:
from synthesis.grammar import *
from synthesis.synth import *
from synthesis import *

problem = make_rsp_swap_problem(timeout_ms = 120000, objective_name = 'delay+ctot')


# Construct the grammar.
conj = NonTerminal("Rule", sort = problem.env.bool_sort)
cmp = NonTerminal("Atom", sort = problem.env.bool_sort)

by_name = {symbol.name: symbol for symbol in problem.symbols}

flat = lambda xs: [y for x in xs for y in (flat(x) if isinstance(x, list) else [x])]
names = set(by_name)

comparable_pairs = [
    *[("R_i", "R_j")],
    *[
        pair
        for pair in (("DELAY_i", "DELAY'_i"), ("DELAY_j", "DELAY'_j"), ("CTOT_i", "CTOT'_i"), ("CTOT_j", "CTOT'_j"))
        if pair[0] in names and pair[1] in names
    ],
    *[pair for pair in (("D_i_x", "D_j_x"), ("D_x_i", "D_x_j")) if pair[0] in names and pair[1] in names],
]

grammar = Grammar(
    nonterminals=(conj, cmp),
    terminals=problem.symbols,
    start=conj,
    productions=(
        conj >> (cmp & cmp & cmp & cmp & cmp & cmp & cmp),
        cmp >> Choice(tuple(flat(
            [[by_name[l] <= by_name[r], by_name[r] <= by_name[l], by_name[l].eq(by_name[r])] for l, r in comparable_pairs]
        ))),
    ),
)
log(f"Visualising Context-Free Grammar for SyGuS:\n{grammar.vis()}\n")

# Run Synthesis
result = synthesize_pruning_rule(
    problem,
    grammar=grammar,
    require_nonvacuous=True,
)




[13:42:35] Visualising Context-Free Grammar for SyGuS:
<Rule> ::= <Atom> & <Atom> & <Atom> & <Atom> & <Atom> & <Atom> & <Atom>
<Atom> ::= R_i <= R_j | R_j <= R_i | R_i = R_j | DELAY_i <= DELAY'_i | DELAY'_i <= DELAY_i | DELAY_i = DELAY'_i |
           DELAY_j <= DELAY'_j | DELAY'_j <= DELAY_j | DELAY_j = DELAY'_j | CTOT_i <= CTOT'_i | CTOT'_i <= CTOT_i |
           CTOT_i = CTOT'_i | CTOT_j <= CTOT'_j | CTOT'_j <= CTOT_j | CTOT_j = CTOT'_j | D_i_x <= D_j_x | D_j_x <= D_i_x
           | D_i_x = D_j_x | D_x_i <= D_x_j | D_x_j <= D_x_i | D_x_i = D_x_j

[13:42:35] Invoking Synthesis (This may take some time)
[13:42:35] Symbol Set: [Ri, Rj, DELAYi, DELAY'i, DELAYj, DELAY'j, CTOTi, CTOT'i, CTOTj, CTOT'j, Dix, Djx, Dxi, Dxj]
[13:44:54] Synthesis Complete, Result: (SOLUTION):

(
  (define-fun prune ((R_i Int) (R_j Int) (DELAY_i Int) (|DELAY'_i| Int) (DELAY_j Int) (|DELAY'_j| Int) (CTOT_i Int) (|CTOT'_i| Int) (CTOT_j Int) (|CTOT'_j| Int) (D_i_x Int) (D_j_x Int) (D_x_i Int) (D_x_j Int)) Bool (le